# Embed with any model

**What you'll learn.** embpy exposes ~150 pretrained models — DNA, protein,
molecule, text, and single-cell foundation models, plus prior-knowledge
tables — behind a *single* call, `BioEmbedder.embed(...)`. To switch models you
change exactly two things: the `entity_type` and the `model`. Everything else,
including the AnnData you get back, stays the same.

This notebook does two things:

1. **Lists the whole catalog** so you can see every model available, by family.
2. **Embeds with one model from each family, one at a time**, so you can see the
   exact call each time and copy it for your own data.

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd

from embpy import BioEmbedder

embedder = BioEmbedder(device="auto", organism="human")

## 1. The catalog — every model embpy exposes

`model_catalog()` returns the available models as a table. With
`summary=True` you get one row per family; drop it for one row per model, and
pass a family name to filter. Because it is a DataFrame you can sort, filter and
export it — and it renders properly in a notebook instead of printed text.

The `model` column holds exactly the keys you pass to `embed(..., model=...)`.


In [ ]:
# model_catalog() renders as a table; list_available_models() gives bare strings.
embedder.model_catalog(summary=True)


## 2. Embed with one model from each family

The recipe never changes: put the identifiers you already have into a small
AnnData, call `embed(...)`, and read the vectors back from `.obsm[key]`. Below
we walk one representative model per family. Watch how little differs between
them — only `entity_type` and `model`.

### Prior-knowledge table — `genept`

The simplest family: a fixed lookup table. Gene symbols map straight to a
text-derived knowledge vector. No model, no download — instant.

In [ ]:
genes = ["TP53", "EGFR", "MYC", "BRCA1", "JUN", "STAT1"]
gene_ad = ad.AnnData(
    X=np.zeros((len(genes), 1), dtype=np.float32),
    obs=pd.DataFrame({"symbol": genes}, index=genes),
)
gene_ad = embedder.embed(
    gene_ad,
    entity_type="gene",
    id_type="symbol",
    obs_column="symbol",
    model="genept",
    output="anndata",
    key="X_genept",
)
print("genept:", gene_ad.obsm["X_genept"].shape, "→ one vector per gene")

### Protein language model — `esm2_8M`

Give embpy a protein (here by gene symbol); it resolves the amino-acid
sequence for you and runs ESM-2 over it. The small 8M checkpoint downloads once
(~30 MB) and is cached. Same call shape as above — only `entity_type` and
`model` changed.

In [ ]:
proteins = ["TP53", "EGFR", "BRCA1", "STAT1"]
prot_ad = ad.AnnData(
    X=np.zeros((len(proteins), 1), dtype=np.float32),
    obs=pd.DataFrame({"symbol": proteins}, index=proteins),
)
prot_ad = embedder.embed(
    prot_ad,
    entity_type="protein",
    id_type="symbol",
    obs_column="symbol",
    model="esm2_8M",
    output="anndata",
    key="X_esm2",
    attach_to="obs",
)
print("esm2_8M:", prot_ad.obsm["X_esm2"].shape, "→ one vector per protein")

### Molecule fingerprint — `morgan_fp`

Small molecules go in as SMILES. `morgan_fp` computes a structural fingerprint
locally with RDKit — no download. (Pass `attach_to="obs"` so the fingerprint
lands in `.obsm`.)

You do not have to canonicalise first. embpy keys molecules by their
**canonical** SMILES and does that conversion for you, so the same molecule
written two different ways lands on one row. Caffeine below is passed in Kekulé
form deliberately — a different string from its canonical SMILES — and it still
aligns. The form you passed is kept in `obs["input_smiles"]` when it differs
from the canonical one.

In [ ]:
drugs = {
    "aspirin":  "CC(=O)Oc1ccccc1C(=O)O",
    "caffeine": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",   # Kekulé form, not canonical
    "ethanol":  "CCO",
}
smiles = list(drugs.values())

mol_ad = ad.AnnData(
    X=np.zeros((len(smiles), 1), dtype=np.float32),
    obs=pd.DataFrame({"smiles": smiles}, index=smiles),
)
mol_ad = embedder.embed(
    mol_ad,
    entity_type="molecule",
    id_type="smiles",
    obs_column="smiles",
    model="morgan_fp",
    output="anndata",
    key="X_morgan",
    attach_to="obs",
)
print("morgan_fp:", mol_ad.obsm["X_morgan"].shape, "→ one fingerprint per molecule")
print("keyed by:", mol_ad.uns["X_morgan"]["id_scheme"],
      "— the Kekulé caffeine aligned anyway")

### Text encoder — `minilm_l6_v2`

When your entity is just free text — a description, a protocol, a note — embed
the strings directly. `minilm_l6_v2` is a compact sentence encoder (cached
after the first use). Use the text itself as the row id so the vectors align.

In [ ]:
descriptions = [
    "A tumor suppressor that guards genome integrity.",
    "A receptor tyrosine kinase driving cell proliferation.",
]
text_ad = ad.AnnData(
    X=np.zeros((len(descriptions), 1), dtype=np.float32),
    obs=pd.DataFrame({"text": descriptions}, index=descriptions),
)
text_ad = embedder.embed(
    text_ad,
    entity_type="text",
    id_type="text",
    obs_column="text",
    model="minilm_l6_v2",
    output="anndata",
    key="X_minilm",
    attach_to="obs",
)
print("minilm_l6_v2:", text_ad.obsm["X_minilm"].shape, "→ one vector per text")

### DNA sequence model — `hyenadna_small_32k`

The same genes as our very first example, but embedded from their **genomic
sequence** instead of a knowledge table. embpy fetches each gene's DNA from
Ensembl and runs a long-context HyenaDNA model — so this is the one cell that
reaches the network at run time. If Ensembl is briefly unavailable it may error;
just re-run the cell (the downloaded model itself is cached).

In [ ]:
dna_ad = ad.AnnData(
    X=np.zeros((len(genes), 1), dtype=np.float32),
    obs=pd.DataFrame({"symbol": genes}, index=genes),
)
dna_ad = embedder.embed(
    dna_ad,
    entity_type="gene",
    id_type="symbol",
    obs_column="symbol",
    model="hyenadna_small_32k",
    region="exons",     # full genomic loci run to ~200 kb, past this model's 32 k context
    missing="nan",      # Ensembl is occasionally flaky; NaN-fill rather than abort
    output="anndata",
    key="X_hyenadna",
    attach_to="obs",
)
resolved = int(np.isfinite(dna_ad.obsm["X_hyenadna"]).all(axis=1).sum())
print(f"hyenadna_small_32k: {dna_ad.obsm['X_hyenadna'].shape}"
      f"  ({resolved}/{len(genes)} genes resolved via Ensembl)")


### Single-cell & morphology models

The two remaining families work on *data matrices*, not identifier lists.
Single-cell models (`scvi`, `geneformer`, …) embed a whole cell × gene count
matrix; morphology models embed images. The `embed()` entry point is the same,
but they need real data, so they get their own tutorials — see
[Embedding cells](cells.ipynb). The call looks like:

```python
cells = ad.read_h5ad("my_counts.h5ad")     # cells × genes
cells = embedder.embed(cells, entity_type="cell", model="scvi",
                       output="anndata", key="X_scvi")
```

## Recap

Six families, one call. Only `entity_type` and `model` ever changed:

| Family | `entity_type` | `model` | You passed |
| --- | --- | --- | --- |
| prior knowledge | `"gene"` | `genept` | gene symbols |
| protein LM | `"protein"` | `esm2_8M` | gene/protein symbols |
| molecule | `"molecule"` | `morgan_fp` | SMILES, in any equivalent form |
| text | `"text"` | `minilm_l6_v2` | free-text strings |
| DNA sequence | `"gene"` | `hyenadna_small_32k` | gene symbols |
| single cell | `"cell"` | `scvi` | a count matrix |

In every case you passed the identifiers you already had, and embpy resolved
them to its own canonical scheme — gene symbols to Ensembl ids, SMILES to
canonical SMILES — so you never had to normalise anything first.

Every call returned your AnnData with the embedding in `.obsm[key]` and its
provenance in `.uns` — `.X` is never touched.

**Next:** [Where embeddings live](02_output_contract.ipynb) explains that
storage contract in full, and [Comparing embeddings](03_compare_embeddings.ipynb)
shows how to tell whether two of these models see your biology the same way.